In [63]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from openai import OpenAI

In [64]:
kundeklager_df = pd.read_csv('Kundeklager.csv')

C:\Users\ilebr\AppData\Local\Temp\ipykernel_42896\1501739571.py:1: DtypeWarning: Columns (0: Sub-issue, 1: Consumer complaint narrative, 2: Company public response, 3: Consumer consent provided?, 4: Consumer disputed?) have mixed types. Specify dtype option on import or set low_memory=False.
  kundeklager_df = pd.read_csv('Kundeklager.csv')


In [65]:
kundeklager_df.shape

(1282355, 18)

In [66]:
kolonner = kundeklager_df.columns
print(kolonner)

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='str')


In [67]:
kundeklager_df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,05/10/2019,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,NAVY FEDERAL CREDIT UNION,FL,328XX,Older American,NaN,Web,05/10/2019,In progress,Yes,NaN,3238275
1,05/10/2019,Checking or savings account,Other banking product or service,Managing an account,Deposits and withdrawals,NaN,NaN,BOEING EMPLOYEES CREDIT UNION,WA,98204,NaN,NaN,Referral,05/10/2019,Closed with explanation,Yes,NaN,3238228
2,05/10/2019,Debt collection,Payday loan debt,Communication tactics,Frequent or repeated calls,NaN,NaN,CURO Intermediate Holdings,TX,751XX,NaN,NaN,Web,05/10/2019,Closed with explanation,Yes,NaN,3237964
3,05/10/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Old information reappears or never goes away,NaN,NaN,Ad Astra Recovery Services Inc,LA,708XX,NaN,NaN,Web,05/10/2019,Closed with explanation,Yes,NaN,3238479
4,05/10/2019,Checking or savings account,Checking account,Managing an account,Banking errors,NaN,NaN,ALLY FINANCIAL INC.,AZ,85205,NaN,NaN,Postal mail,05/10/2019,In progress,Yes,NaN,3238460


In [68]:
# Sjekker at det ikke er dupliakter i datasettet
print(kundeklager_df["Complaint ID"].nunique())

1282355


In [69]:
# Kun fritekst
kundeklager_df = kundeklager_df[kundeklager_df["Consumer complaint narrative"].notna()]
print(len(kundeklager_df))   

383564


In [70]:
# Bare produktfamilien "Credit card or prepaid card"
kundeklager_df = kundeklager_df[kundeklager_df["Product"] == "Credit card or prepaid card"]
print(len(kundeklager_df))   

21379


In [71]:
# Ser alle kategoriene (issues) innad i produktfamilien
print(kundeklager_df["Issue"].value_counts())

Issue
Problem with a purchase shown on your statement                                     4604
Other features, terms, or problems                                                  3060
Fees or interest                                                                    2994
Problem when making payments                                                        2208
Closing your account                                                                1716
Getting a credit card                                                               1715
Advertising and marketing, including promotional offers                             1691
Trouble using your card                                                              877
Incorrect information on your report                                                 528
Problem with a purchase or transfer                                                  468
Struggling to pay your bill                                                          372
Trouble using t

In [72]:
# Velger de 7 største kategoriene
kundeklager_df = kundeklager_df['Issue'].value_counts().head(7).index

In [73]:
data = kredittkort[kredittkort["Issue"].isin(topp7)]

print("Klager i de sju største kategoriene:", len(data))   # skal gi 17 988
print("Andel:", round(len(data) / len(kredittkort) * 100, 1), "%")   # skal gi 84.1 %
print("Utelatte kategorier:", kredittkort["Issue"].nunique() - 7)   # skal gi 15

Klager i de sju største kategoriene: 17988
Andel: 84.1 %
Utelatte kategorier: 15


In [74]:
print(data["Issue"].value_counts())

Issue
Problem with a purchase shown on your statement            4604
Other features, terms, or problems                         3060
Fees or interest                                           2994
Problem when making payments                               2208
Closing your account                                       1716
Getting a credit card                                      1715
Advertising and marketing, including promotional offers    1691
Name: count, dtype: int64
